# Satellite data and orbital mechanics exploration

## Project context
This notebook presents our end-to-end satellite data processing pipeline. We combine orbital mechanics calculations with Earth observation imagery over Kourou, French Guiana.

We will follow four main steps:

1. **Orbital data ingestion:** Load and parse TLE (Two-Line Element) data for satellite tracking.
2. **Earth observation ingestion:** Connect to the STAC API to search and download Sentinel-2 images over Kourou CSG.
3. **Data visualization:** Render satellite imagery using Matplotlib and Pillow.
4. **Spectral analysis:** Perform RGB distribution analysis with NumPy.

In [1]:
import os
import sys

# Add project root directory to sys.path for internal imports
sys.path.append(os.path.abspath(".."))

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Setup data storage path
DATA_DIR = "../data/raw"
os.makedirs(DATA_DIR, exist_ok=True)
IMAGE_PATH = os.path.join(DATA_DIR, "sentinel_kourou.jpg")

## 1 Orbital mechanics and TLE data loading
In this section, we load Two-Line Element (TLE) orbital data for the International Space Station (ISS) from a raw text file to extract orbital parameters.

### 1.1 TLE data ingestion

* Target object: International Space Station (ISS)
* Data format: Two-Line Element Set (TLE)
* Source: Official orbital parameters retrieved from [CelesTrak](https://celestrak.org/)

In [2]:
# Import TLE parser function from internal project module
from src.ingestion.load_tle import load_tle

# Load TLE lines from raw data directory
tle_file_path = os.path.join(DATA_DIR, "iss.txt")
lines = load_tle(tle_file_path)

# Display loaded lines count and content
print(f"Loaded {len(lines)} lines from TLE file.\n")
for line in lines:
    print(line)

Loaded 3 lines from TLE file.

ISS (ZARYA)             

1 25544U 98067A   26270.17419514  .00009528  00000+0  18291-3 0  9996

2 25544  51.6315 155.3455 0007168 193.0560 167.0244 15.48664528587561



### 1.2 Real-time orbit propagation and mapping

We compute the current ISS orbital position using SGP4 propagation and Astropy coordinate transformations, then render the result on an interactive map.

* Propagation model: SGP4 (Simplified General Perturbations Model) via [SGP4 Python library](https://pypi.org/project/sgp4/)
* Coordinate transformations: TEME to Earth-fixed ITRS conversion via [Astropy Coordinates](https://docs.astropy.org/en/stable/coordinates/)
* Base map tiles: Esri World Imagery ([ArcGIS Online](https://www.arcgis.com/))

In [3]:
from src.orbital import generate_iss_map

# Generate and display the interactive map
iss_map = generate_iss_map()
iss_map

Time (UTC): 2026-10-23 06:49:04.563
Latitude:   -47.0243°
Longitude:  129.8172°
Altitude:   431.35 km km


## 2. Earth observation data ingestion (Sentinel-2)

In this section, we connect to the SpatioTemporal Asset Catalog (STAC) API to search for Sentinel-2 satellite imagery over the Guiana Space Centre (CSG) and download the lowest cloud-cover scene.

### 2.1 STAC API ingestion and target location

* Target Site: Guiana Space Centre (CSG), Kourou, French Guiana
* Coordinates: Longitude: -52.763674°, Latitude: 5.189573°
* Coordinates Source: Official GPS location retrieved from [Géoportail (IGN France)](https://www.geoportail.gouv.fr/)
* Data Catalog: [Earth Search STAC API by Element 84](https://element84.com/earth-search/) (Open Data)
* Collection: `sentinel-2-l2a` (Level-2A Surface Reflectance)
* Python API Client: [pystac-client](https://pystac-client.readthedocs.io/)

We define a bounding box (~2 km x 2 km) around these coordinates to query low-cloud-cover scenes.

In [4]:
# Import ingestion functions from source package
from src.ingestion.sentinel import search_sentinel_images, download_preview_image

# CSG coordinates
csg_lon, csg_lat = -52.763674, 5.189573
ariane6_bbox = [csg_lon - 0.01, csg_lat - 0.01, csg_lon + 0.01, csg_lat + 0.01]

# Query STAC API for Sentinel-2 scenes
res = search_sentinel_images(ariane6_bbox, date_range="2026-01-01/..", max_cloud_cover=10, limit=3)

# Download the best available image
if res:
    download_preview_image(res[0], IMAGE_PATH)

Downloading: https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/22/N/BL/2026/10/S2C_22NBL_20261007_0_L2A/preview.jpg
Saved: ../data/raw\sentinel_kourou.jpg


## References and documentation

### Data and services
- **Geographical location:** [Géoportail IGN France](https://www.geoportail.gouv.fr/)
- **Satellite TLE data:** [CelesTrak Orbital Data](https://celestrak.org/)
- **STAC catalog and API:** [Element 84 Earth Search Docs](https://element84.com/earth-search/)
- **STAC endpoint:** [Element 84 AWS API](https://earth-search.aws.element84.com/v1)
- **Earth observation imagery:** Copernicus Sentinel-2 L2A (European Space Agency — ESA)

### Python libraries documentation
- **Orbital propagation:** [SGP4 Documentation](https://pypi.org/project/sgp4/)
- **Coordinate transformations:** [Astropy Coordinates Documentation](https://docs.astropy.org/en/stable/coordinates/)
- **Interactive mapping:** [Folium Documentation](https://python-visualization.github.io/folium/)
- **STAC client:** [pystac-client Documentation](https://pystac-client.readthedocs.io/)
- **HTTP requests:** [Requests Documentation](https://requests.readthedocs.io/)
- **Image processing:** [Pillow (PIL) Documentation](https://pillow.readthedocs.io/)
- **Data analysis:** [NumPy Documentation](https://numpy.org/doc/)
- **Visualization:** [Matplotlib Documentation](https://matplotlib.org/stable/contents.html)